In [1]:
import pandas as pd

`clean-weather.csv` contains the weather/wildfire data after having being thoroughly processed and cleaned in `data-cleaning-preparation.ipnyb`.

In [2]:
df = pd.read_csv('clean-weather.csv')

In [3]:
df.head()

,LATITUDE,LONGITUDE,ELEVATION,STATION_NAME,PROVINCE_CODE,ID,LOCAL_DATE,LOCAL_DAY,LOCAL_MONTH,LOCAL_YEAR,MIN_TEMPERATURE,MAX_TEMPERATURE,MEAN_TEMPERATURE,SNOW_ON_GROUND,TOTAL_SNOW,TOTAL_PRECIPITATION,TOTAL_RAIN,hectares
0,50.9050,-126.9292,14.0,EGG ISLAND,BC,1062646.2000.3.10,2000-03-10 00:00:00,10.0,3.0,2000.0,2.0,11.3,6.7,0.0,0.0,3.5,3.5,6.0
1,59.9617,-121.3608,498.0,SAMBAA K'E,NT,220CQHR.2000.6.27,2000-06-27 00:00:00,27.0,6.0,2000.0,7.9,27.0,17.5,0.0,0.0,0.0,0.0,1000.0
2,59.9752,-121.0342,498.0,SAMBAA K'E,NT,220CQHR.2000.7.11,2000-07-11 00:00:00,11.0,7.0,2000.0,9.9,20.9,15.4,0.0,0.0,0.0,0.0,12.0
3,59.1767,-122.0190,378.3,FORT NELSON UA,BC,1192950.2001.6.13,2001-06-13 00:00:00,13.0,6.0,2001.0,8.5,26.8,17.7,0.0,0.0,0.0,0.0,5.0
4,59.4008,-120.6438,777.2,PETITOT LO,AB,3075171.2000.6.26,2000-06-26 00:00:00,26.0,6.0,2000.0,8.0,24.0,16.0,0.0,0.0,1.2,1.2,0.1


In [4]:
df.describe()

,LATITUDE,LONGITUDE,ELEVATION,LOCAL_DAY,LOCAL_MONTH,LOCAL_YEAR,MIN_TEMPERATURE,MAX_TEMPERATURE,MEAN_TEMPERATURE,SNOW_ON_GROUND,TOTAL_SNOW,TOTAL_PRECIPITATION,TOTAL_RAIN,hectares
count,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000,57666.000000
mean,52.443934,-102.165135,502.642020,15.491624,5.842368,2009.088787,4.925190,18.486696,11.721789,2.615430,0.101623,1.678459,1.135021,257.431717
std,4.707561,20.771427,392.902579,8.822762,1.728554,6.648006,8.582523,9.605800,8.742616,11.997669,1.026405,4.787095,4.034979,5361.099326
min,40.065300,-162.480000,1.200000,1.000000,1.000000,1999.000000,-46.900000,-42.700000,-44.800000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,49.210100,-117.763088,224.100000,8.000000,5.000000,2004.000000,0.000000,13.000000,6.900000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,52.031100,-111.272683,419.000000,15.000000,6.000000,2008.000000,6.400000,20.500000,13.500000,0.000000,0.000000,0.000000,0.000000,0.009000
75%,55.500000,-84.569900,690.400000,23.000000,7.000000,2014.000000,11.000000,25.500000,18.000000,0.000000,0.000000,0.900000,0.000000,0.200000
max,82.310400,116.188000,2926.100000,31.000000,12.000000,2024.000000,26.500000,47.900000,34.400000,319.000000,37.000000,168.000000,168.000000,577646.800000


# Classification
We turn this into a classification problem by creating a class column, where 0 means no fire and 1 means fire

In [5]:
df['CLASS'] = (df.loc[:, 'hectares'] > 0).astype(int)

# Modeling
It is not evident which model would best be used for our dataset, so we perform simple tests to see if basic models might be viable using a single test-train split

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [7]:
# x = df[['LATITUDE', 'LONGITUDE', 'ELEVATION', 'MEAN_TEMPERATURE', 'MAX_TEMPERATURE', 'MIN_TEMPERATURE', 'SNOW_ON_GROUND', 'TOTAL_SNOW', 'TOTAL_PRECIPITATION', 'TOTAL_RAIN', 'MIN_REL_HUMIDITY', 'MAX_REL_HUMIDITY', 'SPEED_MAX_GUST']] # Only select weather data
df.dropna(subset=['ELEVATION'], inplace=True)
x = df[['LATITUDE', 'LONGITUDE', 'ELEVATION', 'MEAN_TEMPERATURE', 'MAX_TEMPERATURE', 'MIN_TEMPERATURE', 'SNOW_ON_GROUND', 'TOTAL_SNOW', 'TOTAL_PRECIPITATION', 'TOTAL_RAIN']] # Only select weather data
y = df['CLASS']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

## Logistic Regression model

In [8]:
from sklearn.linear_model import LogisticRegression

In [9]:
model = LogisticRegression()
model.fit(x_train, y_train)

C:\Users\ethan\PycharmProjects\fire-guard\model\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [10]:
# Predicting on the train set
y_pred_train = model.predict(x_train)
# Evaluating the model on the train set
accuracy_train = accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train)
recall_train = recall_score(y_train, y_pred_train)
f1_train = f1_score(y_train, y_pred_train)
print(f"Accuracy: {accuracy_train}")
print(f"Precision: {precision_train}")
print(f"Recall: {recall_train}")
print(f"f1: {f1_train}")

Accuracy: 0.6855978496488337
Precision: 0.6712337790659377
Recall: 0.8184081599871497
f1: 0.7375506658946149


In [11]:
# Predicting on the test set
y_pred_test = model.predict(x_test)
accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Recall: {recall_test}")
print(f"f1: {f1_test}")

Accuracy: 0.6844113057048725
Precision: 0.6695228051674137
Recall: 0.8176110753380553
f1: 0.7361936512538049


## Decision Tree Classifier

In [12]:
from sklearn.tree import DecisionTreeClassifier

In [13]:
model = DecisionTreeClassifier(random_state=11)
model.fit(x_train, y_train)

DecisionTreeClassifier(random_state=11)

In [14]:
# Predicting on the train set
y_pred_train = model.predict(x_train)

# Evaluating the model on the train set
accuracy_train = accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train)
recall_train = recall_score(y_train, y_pred_train)
f1_train = f1_score(y_train, y_pred_train)

print(f"Accuracy: {accuracy_train}")
print(f"Precision: {precision_train}")
print(f"Recall: {recall_train}")
print(f"f1: {f1_train}")

Accuracy: 0.9999349692187636
Precision: 1.0
Recall: 0.9998795277487752
f1: 0.9999397602457782


In [15]:
# Predicting on the test set
y_pred_test = model.predict(x_test)
accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Recall: {recall_test}")
print(f"f1: {f1_test}")

Accuracy: 0.8311947286284029
Precision: 0.8490751350466524
Recall: 0.8349967804249839
f1: 0.8419771122473825


Decision Tree Regressor has overfit

## Random Forests 

In [16]:
from sklearn.ensemble import RandomForestClassifier

In [17]:
model = RandomForestClassifier()
model.fit(x_train, y_train)

RandomForestClassifier()

In [18]:
# Predicting on the train set
y_pred_train = model.predict(x_train)

# Evaluating the model on the train set
accuracy_train = accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train)
recall_train = recall_score(y_train, y_pred_train)
f1_train = f1_score(y_train, y_pred_train)

print(f"Accuracy: {accuracy_train}")
print(f"Precision: {precision_train}")
print(f"Recall: {recall_train}")
print(f"f1: {f1_train}")

Accuracy: 0.9999132922916847
Precision: 0.9999598393574297
Recall: 0.9998795277487752
f1: 0.9999196819404843


In [19]:
# Predicting on the test set
y_pred_test = model.predict(x_test)
accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Recall: {recall_test}")
print(f"f1: {f1_test}")

Accuracy: 0.8357898387376452
Precision: 0.8464377406931964
Recall: 0.8491629104958146
f1: 0.8477981356477017


## Gradient Boosting Model

In [20]:
from sklearn.ensemble import GradientBoostingClassifier

In [21]:
model = GradientBoostingClassifier()
model.fit(x_train, y_train)

GradientBoostingClassifier()

In [22]:
# Predicting on the train set
y_pred_train = model.predict(x_train)

# Evaluating the model on the train set
accuracy_train = accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train)
recall_train = recall_score(y_train, y_pred_train)
f1_train = f1_score(y_train, y_pred_train)

print(f"Accuracy: {accuracy_train}")
print(f"Precision: {precision_train}")
print(f"Recall: {recall_train}")
print(f"f1: {f1_train}")

Accuracy: 0.7745382814532212
Precision: 0.7678525250286306
Recall: 0.8346719139024978
f1: 0.799869157799542


In [23]:
# Predicting on the test set
y_pred_test = model.predict(x_test)
accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Recall: {recall_test}")
print(f"f1: {f1_test}")

Accuracy: 0.7653892838564245
Precision: 0.759089565474431
Recall: 0.8267868641339343
f1: 0.7914932963476653


## XGB Classifier

In [24]:
from xgboost import XGBClassifier

In [25]:
model = XGBClassifier()
model.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [26]:
# Predicting on the train set
y_pred_train = model.predict(x_train)

# Evaluating the model on the train set
accuracy_train = accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train)
recall_train = recall_score(y_train, y_pred_train)
f1_train = f1_score(y_train, y_pred_train)

print(f"Accuracy: {accuracy_train}")
print(f"Precision: {precision_train}")
print(f"Recall: {recall_train}")
print(f"f1: {f1_train}")

Accuracy: 0.9092387063209919
Precision: 0.9203360253236476
Recall: 0.9106899044253474
f1: 0.9154875562642553


In [27]:
# Predicting on the test set
y_pred_test = model.predict(x_test)

accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Recall: {recall_test}")
print(f"f1: {f1_test}")

Accuracy: 0.8850355470782035
Precision: 0.896076523994812
Recall: 0.8897295556986478
f1: 0.892891760904685


## K-Nearest Neighbors

In [28]:
from sklearn.neighbors import KNeighborsClassifier

In [29]:
model = KNeighborsClassifier()
model.fit(x_train, y_train)

KNeighborsClassifier()

In [30]:
# Predicting on the train set
y_pred_train = model.predict(x_train)

# Evaluating the model on the train set
accuracy_train = accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train)
recall_train = recall_score(y_train, y_pred_train)
f1_train = f1_score(y_train, y_pred_train)

print(f"Accuracy: {accuracy_train}")
print(f"Precision: {precision_train}")
print(f"Recall: {recall_train}")
print(f"f1: {f1_train}")

Accuracy: 0.8066201335298708
Precision: 0.8021325670208341
Recall: 0.8518994458276443
f1: 0.8262673080293677


In [31]:
# Predicting on the test set
y_pred_test = model.predict(x_test)

accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Recall: {recall_test}")
print(f"f1: {f1_test}")

Accuracy: 0.7150164730362407
Precision: 0.7188388448301661
Recall: 0.7733419188667096
f1: 0.745094998061264


## Support Vector Machines

In [32]:
from sklearn.svm import SVC

In [33]:
model = SVC()
model.fit(x_train, y_train)

SVC()

In [34]:
# Predicting on the train set
y_pred_train = model.predict(x_train)

# Evaluating the model on the train set
accuracy_train = accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train)
recall_train = recall_score(y_train, y_pred_train)
f1_train = f1_score(y_train, y_pred_train)

print(f"Accuracy: {accuracy_train}")
print(f"Precision: {precision_train}")
print(f"Recall: {recall_train}")
print(f"f1: {f1_train}")

Accuracy: 0.6861180958987254
Precision: 0.662013431165278
Recall: 0.8550718817765641
f1: 0.7462587179756773


In [35]:
# Predicting on the test set
y_pred_test = model.predict(x_test)

accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Recall: {recall_test}")
print(f"f1: {f1_test}")

Accuracy: 0.6838044043696896
Precision: 0.6591783542261388
Recall: 0.8549581455247908
f1: 0.7444109608241642
